In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score


In [ ]:
# ======================
# 1. ПАРАМЕТРЫ
# ======================
DATA = Path("../data/titanic.csv")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
TEST_SIZE = 0.25


In [ ]:
data = pd.read_csv(DATA)


In [ ]:
print("Number of rows:", data.shape[0])
print("Columns:", list(data.columns))
data.head()


In [ ]:
# Пропуски
print(data.isna().sum().sort_values(ascending=False).head(8))
print('survived rate:', round(data['Survived'].mean(), 3))


In [ ]:
# Survive rate by Sex / Pclass
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
data.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[0], color='#21918c', title='Survive by Sex')
data.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[1], color='#440154', title='Survive by Pclass')
for ax in axes:
    ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(FIG / 'titanic_pipeline_eda.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
data.info()


In [ ]:
# ======================
# 2. ЛЁГКИЙ FEATURE ENGINEERING
# ======================
df = data.copy()
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

num_cols = ["Age", "Fare", "FamilySize", "IsAlone"]
cat_cols = ["Pclass", "Sex", "Embarked"]
target = "Survived"

print(df[["FamilySize", "IsAlone"]].describe().round(2))


In [ ]:
# Распределение размера семьи
plt.figure(figsize=(8, 5))
sns.histplot(df["FamilySize"], bins=range(1, 12), kde=False, color="steelblue", edgecolor="white")
plt.title("Распределение FamilySize", fontsize=14, pad=15)
plt.xlabel("Размер семьи", fontsize=12)
plt.ylabel("Количество пассажиров", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(FIG / "titanic_family_size.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ======================
# 3. TRAIN / TEST
# ======================
X = df[num_cols + cat_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
X_train.shape, X_test.shape


In [ ]:
# ======================
# 4. ФУНКЦИЯ СБОРКИ ПАЙПЛАЙНА
# ======================
def make_pipe(model, scale_numeric=True):
    # Числа: imputer (+ scaler для линейных моделей)
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))

    prep = ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ])
    return Pipeline([("prep", prep), ("model", model)])


In [ ]:
# ======================
# 5. КАНДИДАТЫ МОДЕЛЕЙ
# ======================
candidates = {
    "logreg": make_pipe(LogisticRegression(max_iter=1000), scale_numeric=True),
    "rf": make_pipe(
        RandomForestClassifier(n_estimators=300, max_depth=6, random_state=RANDOM_SEED),
        scale_numeric=False,
    ),
    "gb": make_pipe(GradientBoostingClassifier(random_state=RANDOM_SEED), scale_numeric=False),
}
list(candidates.keys())


In [ ]:
# ======================
# 6. CV + HOLDOUT
# ======================
rows = []
for name, pipe in candidates.items():
    # CV на train — оценка без подглядывания в test
    cv = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc")
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    rows.append({
        "model": name,
        "cv_auc_mean": round(cv.mean(), 3),
        "cv_auc_std": round(cv.std(), 3),
        "test_auc": round(roc_auc_score(y_test, proba), 3),
    })

results = pd.DataFrame(rows).sort_values("test_auc", ascending=False)
results


In [ ]:
# ======================
# 7. ВИЗУАЛИЗАЦИЯ СРАВНЕНИЯ
# ======================
plot_df = results.set_index("model")[["cv_auc_mean", "test_auc"]]

plt.figure(figsize=(9, 5))
plot_df.plot(kind="bar", ax=plt.gca(), color=["#440154", "#21918c"], edgecolor="white")
plt.title("ROC-AUC: CV mean vs holdout", fontsize=14, pad=15)
plt.ylabel("ROC-AUC", fontsize=12)
plt.xlabel("Модель", fontsize=12)
plt.xticks(rotation=0)
plt.ylim(0.7, 1.0)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.legend(title="")
plt.tight_layout()
plt.savefig(FIG / "titanic_pipeline_auc.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
print("best on test:", results.iloc[0].to_dict())
print("Берём модель с лучшим test_auc при адекватном CV.")
